Étape 2 — Scoring pédagogique et calcul des niveaux
Attention : cette étape nécessite les réponses correctes / barèmes des tests.
Sans barème, le code peut préparer les colonnes et la structure, mais il ne peut pas calculer un score fiable.

Donc l’objectif de cette Étape 2 sont:
1. charger les fichiers préparés de l’Étape 1 ;
2. détecter les questions ;
3. créer les dictionnaires de correction ;
4. calculer les scores initiaux par domaine ;
5. calculer les scores intermédiaires ;
6. calculer les scores finaux ;
7. calculer les pourcentages ;
8. attribuer les niveaux : Débutant, Intermédiaire, Avancé ;
9. calculer la progression ;
10. exporter les datasets scorés.

# # Cellule 1 — Importation des bibliothèques

In [1]:
# ============================================================
# ÉTAPE 2 — SCORING PÉDAGOGIQUE ET CALCUL DES NIVEAUX
# Projet : Prédiction et évaluation de la performance académique
# ============================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

# Cellule 2 — Définition des chemins

In [2]:
# ============================================================
# 2.1. Chemins des fichiers préparés
# ============================================================

INPUT_DIR = Path("outputs/01_integration_preparation")
OUTPUT_DIR = Path("outputs/02_scoring_pedagogique")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_INSCRIPTIONS = INPUT_DIR / "01_inscriptions_preparees.csv"
PATH_TEST_INITIAL = INPUT_DIR / "02_test_initial_prepare.csv"
PATH_TESTS_INTERMEDIAIRES = INPUT_DIR / "03_tests_intermediaires_prepares.csv"
PATH_TESTS_FINAUX = INPUT_DIR / "04_tests_finaux_prepares.csv"

PATH_DATASET_ORIENTATION = INPUT_DIR / "05_dataset_orientation_initiale.csv"
PATH_DATASET_PREDICTION = INPUT_DIR / "06_dataset_prediction_finale_pre_scoring.csv"
PATH_DATASET_LONGITUDINAL = INPUT_DIR / "07_dataset_longitudinal_pre_scoring.csv"

fichiers = {
    "inscriptions": PATH_INSCRIPTIONS,
    "test_initial": PATH_TEST_INITIAL,
    "tests_intermediaires": PATH_TESTS_INTERMEDIAIRES,
    "tests_finaux": PATH_TESTS_FINAUX,
    "dataset_orientation": PATH_DATASET_ORIENTATION,
    "dataset_prediction": PATH_DATASET_PREDICTION,
    "dataset_longitudinal": PATH_DATASET_LONGITUDINAL
}

for nom, chemin in fichiers.items():
    print(f"{nom:25s} | existe = {chemin.exists()} | {chemin}")

inscriptions              | existe = True | outputs/01_integration_preparation/01_inscriptions_preparees.csv
test_initial              | existe = True | outputs/01_integration_preparation/02_test_initial_prepare.csv
tests_intermediaires      | existe = True | outputs/01_integration_preparation/03_tests_intermediaires_prepares.csv
tests_finaux              | existe = True | outputs/01_integration_preparation/04_tests_finaux_prepares.csv
dataset_orientation       | existe = True | outputs/01_integration_preparation/05_dataset_orientation_initiale.csv
dataset_prediction        | existe = True | outputs/01_integration_preparation/06_dataset_prediction_finale_pre_scoring.csv
dataset_longitudinal      | existe = True | outputs/01_integration_preparation/07_dataset_longitudinal_pre_scoring.csv


# Cellule 3 — Chargement des datasets

In [3]:
# ============================================================
# 2.2. Chargement des données préparées
# ============================================================

df_inscriptions = pd.read_csv(PATH_INSCRIPTIONS)
df_test_initial = pd.read_csv(PATH_TEST_INITIAL)
df_tests_intermediaires = pd.read_csv(PATH_TESTS_INTERMEDIAIRES)
df_tests_finaux = pd.read_csv(PATH_TESTS_FINAUX)

df_orientation = pd.read_csv(PATH_DATASET_ORIENTATION)
df_prediction = pd.read_csv(PATH_DATASET_PREDICTION)
df_longitudinal = pd.read_csv(PATH_DATASET_LONGITUDINAL)

print("Chargement terminé.")
print("Inscriptions :", df_inscriptions.shape)
print("Test initial :", df_test_initial.shape)
print("Tests intermédiaires :", df_tests_intermediaires.shape)
print("Tests finaux :", df_tests_finaux.shape)
print("Dataset orientation :", df_orientation.shape)
print("Dataset prédiction :", df_prediction.shape)
print("Dataset longitudinal :", df_longitudinal.shape)

Chargement terminé.
Inscriptions : (472, 20)
Test initial : (305, 23)
Tests intermédiaires : (196, 20)
Tests finaux : (206, 23)
Dataset orientation : (305, 42)
Dataset prédiction : (128, 64)
Dataset longitudinal : (125, 84)


# Cellule 4 — Fonctions générales de scoring

In [4]:
# ============================================================
# 2.3. Fonctions générales de scoring
# ============================================================

def normaliser_reponse(valeur):
    """
    Normalise une réponse pour faciliter la comparaison :
    - conversion en texte ;
    - suppression des espaces multiples ;
    - mise en minuscules ;
    - gestion des valeurs manquantes.
    """
    if pd.isna(valeur):
        return np.nan
    
    valeur = str(valeur).strip()
    valeur = re.sub(r"\s+", " ", valeur)
    
    if valeur.lower() in ["", "nan", "none", "nat", "null"]:
        return np.nan
    
    return valeur.lower()


def est_reponse_inconnue(valeur):
    """
    Détecte les réponses inconnues ou non renseignées.
    """
    if pd.isna(valeur):
        return True
    
    valeur_norm = normaliser_reponse(valeur)
    
    if pd.isna(valeur_norm):
        return True
    
    motifs_inconnus = [
        "je ne sais pas",
        "ne sais pas",
        "aucune réponse",
        "sans réponse",
        "non répondu",
        "non repondu",
        "inconnu",
        "unknown"
    ]
    
    return any(motif in valeur_norm for motif in motifs_inconnus)


def corriger_reponse(reponse, bonne_reponse):
    """
    Retourne :
    - 1 si la réponse est correcte ;
    - 0 si la réponse est incorrecte ;
    - 0 si la réponse est inconnue ou manquante.
    
    NB : dans cette version, une réponse manquante est considérée comme incorrecte.
    """
    if est_reponse_inconnue(reponse):
        return 0
    
    if bonne_reponse is None or pd.isna(bonne_reponse):
        return np.nan
    
    reponse_norm = normaliser_reponse(reponse)
    bonne_reponse_norm = normaliser_reponse(bonne_reponse)
    
    if pd.isna(reponse_norm) or pd.isna(bonne_reponse_norm):
        return 0
    
    return int(reponse_norm == bonne_reponse_norm)


def attribuer_niveau_depuis_pourcentage(score_pct):
    """
    Attribue un niveau de compétence selon le score en pourcentage.
    
    Règle officielle du mémoire :
    - Débutant : score < 50 %
    - Intermédiaire : 50 % <= score < 70 %
    - Avancé : score >= 70 %
    """
    if pd.isna(score_pct):
        return np.nan
    
    if score_pct < 50:
        return "Débutant"
    elif score_pct < 70:
        return "Intermédiaire"
    else:
        return "Avancé"


def afficher_shape(nom, df):
    print(f"{nom:45s} : {df.shape[0]} lignes | {df.shape[1]} colonnes")

# Cellule 5— Détection numérique correcte Q1 à Q20

In [5]:
# ============================================================
# 2.4. Détection des colonnes de questions — VERSION CORRIGÉE
# ============================================================

def extraire_numero_question_initiale(colonne):
    """
    Extrait correctement le numéro d'une question initiale.
    Exemples :
    - Q1 - Resume ventes Excel  -> 1
    - Q10 - 12 commerciaux      -> 10
    - Q20 - Sortie fiable LLM   -> 20
    """
    col = str(colonne).strip()
    match = re.match(r"^Q\s*(\d{1,2})(\s|-|_|:|\.|$)", col, flags=re.IGNORECASE)
    
    if match:
        numero = int(match.group(1))
        if 1 <= numero <= 20:
            return numero
    
    return None


def detecter_questions_initiales(df):
    """
    Détecte et trie correctement les colonnes Q1 à Q20.
    """
    colonnes_questions = []
    
    for col in df.columns:
        numero = extraire_numero_question_initiale(col)
        if numero is not None:
            colonnes_questions.append((numero, col))
    
    colonnes_questions = sorted(colonnes_questions, key=lambda x: x[0])
    
    return [col for numero, col in colonnes_questions]


def detecter_questions_standardisees(df, prefixe):
    """
    Détecte les colonnes standardisées :
    mi_Q01 à mi_Q15
    final_Q01 à final_Q15
    """
    pattern = rf"^{prefixe}_Q(\d+)"
    
    colonnes_questions = []
    
    for col in df.columns:
        match = re.match(pattern, str(col), flags=re.IGNORECASE)
        if match:
            numero = int(match.group(1))
            colonnes_questions.append((numero, col))
    
    colonnes_questions = sorted(colonnes_questions, key=lambda x: x[0])
    
    return [col for numero, col in colonnes_questions]


colonnes_q_initial = detecter_questions_initiales(df_test_initial)
colonnes_q_mi = detecter_questions_standardisees(df_tests_intermediaires, "mi")
colonnes_q_final = detecter_questions_standardisees(df_tests_finaux, "final")

print("Questions initiales détectées :", len(colonnes_q_initial))
for i, col in enumerate(colonnes_q_initial, start=1):
    print(f"Q{i:02d} -> {col}")

print("\nQuestions intermédiaires détectées :", len(colonnes_q_mi))
print(colonnes_q_mi)

print("\nQuestions finales détectées :", len(colonnes_q_final))
print(colonnes_q_final)

Questions initiales détectées : 20
Q01 -> Q1 - Resume ventes Excel
Q02 -> Q2 - Import CSV Excel
Q03 -> Q3 - Modele donnees Excel
Q04 -> Q4 - Mauvaise pratique visu Excel
Q05 -> Q5 - 2e grande valeur Excel
Q06 -> Q6 - Mesure vs Colonne DAX
Q07 -> Q7 - CA annee precedente DAX
Q08 -> Q8 - Vue Modele Power BI
Q09 -> Q9 - Acces directeurs regionaux
Q10 -> Q10 - 12 commerciaux 3 indicateurs
Q11 -> Q11 - Bibliotheque CSV Python
Q12 -> Q12 - Overfitting Underfitting
Q13 -> Q13 - Segmentation 50000 clients
Q14 -> Q14 - Deployer modele Python API
Q15 -> Q15 - Valeurs manquantes 30pc
Q16 -> Q16 - Role system prompt LLM
Q17 -> Q17 - Assistant IA PDF financiers
Q18 -> Q18 - Role embedding dans RAG
Q19 -> Q19 - Agent IA selection outil
Q20 -> Q20 - Sortie fiable LLM tableau

Questions intermédiaires détectées : 15
['mi_Q01', 'mi_Q02', 'mi_Q03', 'mi_Q04', 'mi_Q05', 'mi_Q06', 'mi_Q07', 'mi_Q08', 'mi_Q09', 'mi_Q10', 'mi_Q11', 'mi_Q12', 'mi_Q13', 'mi_Q14', 'mi_Q15']

Questions finales détectées : 15
['f

# Cellule 6  — Répartition correcte par domaine

In [6]:
# ============================================================
# 2.5. Répartition des questions initiales par domaine
# VERSION CORRIGÉE
# ============================================================

# Sécurité : vérifier que les 20 questions sont bien détectées
if len(colonnes_q_initial) != 20:
    raise ValueError(
        f"Erreur : {len(colonnes_q_initial)} questions détectées au lieu de 20."
    )

domaines_initial = {
    "DA": colonnes_q_initial[0:5],     # Q1 à Q5
    "BI": colonnes_q_initial[5:10],    # Q6 à Q10
    "DS": colonnes_q_initial[10:15],   # Q11 à Q15
    "IA": colonnes_q_initial[15:20]    # Q16 à Q20
}

for domaine, colonnes in domaines_initial.items():
    print(f"{domaine} : {len(colonnes)} questions")
    for col in colonnes:
        print("  -", col)

DA : 5 questions
  - Q1 - Resume ventes Excel
  - Q2 - Import CSV Excel
  - Q3 - Modele donnees Excel
  - Q4 - Mauvaise pratique visu Excel
  - Q5 - 2e grande valeur Excel
BI : 5 questions
  - Q6 - Mesure vs Colonne DAX
  - Q7 - CA annee precedente DAX
  - Q8 - Vue Modele Power BI
  - Q9 - Acces directeurs regionaux
  - Q10 - 12 commerciaux 3 indicateurs
DS : 5 questions
  - Q11 - Bibliotheque CSV Python
  - Q12 - Overfitting Underfitting
  - Q13 - Segmentation 50000 clients
  - Q14 - Deployer modele Python API
  - Q15 - Valeurs manquantes 30pc
IA : 5 questions
  - Q16 - Role system prompt LLM
  - Q17 - Assistant IA PDF financiers
  - Q18 - Role embedding dans RAG
  - Q19 - Agent IA selection outil
  - Q20 - Sortie fiable LLM tableau


# Cellule 7  — Modèle de correction du test initial

In [7]:
# ============================================================
# 2.6. Création du modèle de correction du test initial
# VERSION CORRIGÉE
# ============================================================

# Sécurité : vérifier que les domaines sont bien définis
if "domaines_initial" not in globals():
    raise ValueError("La variable domaines_initial n'existe pas. Exécute d'abord la Cellule 6.")

if len(colonnes_q_initial) != 20:
    raise ValueError(
        f"Erreur : {len(colonnes_q_initial)} questions détectées au lieu de 20."
    )


# Construction du modèle de correction
lignes_correction_initiale = []

for domaine, colonnes in domaines_initial.items():
    for col in colonnes:
        numero = extraire_numero_question_initiale(col)
        
        lignes_correction_initiale.append({
            "numero_question": numero,
            "domaine": domaine,
            "question": col,
            "bonne_reponse": ""
        })


df_correction_initiale_template = pd.DataFrame(lignes_correction_initiale)

# Tri de sécurité Q1 -> Q20
df_correction_initiale_template = df_correction_initiale_template.sort_values(
    "numero_question"
).reset_index(drop=True)


# Export du modèle
PATH_CORRECTION_INITIAL_TEMPLATE = OUTPUT_DIR / "template_correction_test_initial.csv"

df_correction_initiale_template.to_csv(
    PATH_CORRECTION_INITIAL_TEMPLATE,
    index=False,
    encoding="utf-8-sig"
)


print("Modèle de correction du test initial généré :")
print(PATH_CORRECTION_INITIAL_TEMPLATE)

print("\nAperçu du modèle de correction :")
display(df_correction_initiale_template)

Modèle de correction du test initial généré :
outputs/02_scoring_pedagogique/template_correction_test_initial.csv

Aperçu du modèle de correction :


,numero_question,domaine,question,bonne_reponse
0,1,DA,Q1 - Resume ventes Excel,
1,2,DA,Q2 - Import CSV Excel,
2,3,DA,Q3 - Modele donnees Excel,
3,4,DA,Q4 - Mauvaise pratique visu Excel,
4,5,DA,Q5 - 2e grande valeur Excel,
5,6,BI,Q6 - Mesure vs Colonne DAX,
6,7,BI,Q7 - CA annee precedente DAX,
7,8,BI,Q8 - Vue Modele Power BI,
8,9,BI,Q9 - Acces directeurs regionaux,
9,10,BI,Q10 - 12 commerciaux 3 indicateurs,


# Cellule 8 — Modèles de correction intermédiaire et final

On utilise les fichiers de mapping créés à l’Étape 1
11_mapping_questions_intermediaires.csv
12_mapping_questions_finales.csv

In [8]:
# ============================================================
# 2.7. Création des modèles de correction intermédiaire et final
# VERSION CORRIGÉE AVEC MAPPING DES QUESTIONS
# ============================================================

PATH_MAPPING_MI = INPUT_DIR / "11_mapping_questions_intermediaires.csv"
PATH_MAPPING_FINAL = INPUT_DIR / "12_mapping_questions_finales.csv"

print("Mapping intermédiaire existe :", PATH_MAPPING_MI.exists())
print("Mapping final existe :", PATH_MAPPING_FINAL.exists())


# ============================================================
# 1. Chargement des mappings
# ============================================================

if not PATH_MAPPING_MI.exists():
    raise FileNotFoundError(
        "Le fichier 11_mapping_questions_intermediaires.csv est introuvable. "
        "Vérifie les exports de l'Étape 1."
    )

if not PATH_MAPPING_FINAL.exists():
    raise FileNotFoundError(
        "Le fichier 12_mapping_questions_finales.csv est introuvable. "
        "Vérifie les exports de l'Étape 1."
    )

df_mapping_questions_intermediaires = pd.read_csv(PATH_MAPPING_MI)
df_mapping_questions_finales = pd.read_csv(PATH_MAPPING_FINAL)

print("\nMapping intermédiaire :", df_mapping_questions_intermediaires.shape)
print("Mapping final :", df_mapping_questions_finales.shape)


# ============================================================
# 2. Création du modèle de correction des tests intermédiaires
# ============================================================

df_correction_mi_template = df_mapping_questions_intermediaires.copy()

df_correction_mi_template = df_correction_mi_template.rename(columns={
    "ancienne_colonne": "question_originale",
    "nouvelle_colonne": "question_standardisee"
})

df_correction_mi_template["type_test"] = "intermediaire"
df_correction_mi_template["numero_question"] = (
    df_correction_mi_template["question_standardisee"]
    .str.extract(r"Q(\d+)", expand=False)
    .astype(int)
)

df_correction_mi_template["bonne_reponse"] = ""

df_correction_mi_template = df_correction_mi_template[
    [
        "type_test",
        "parcours",
        "feuille",
        "numero_question",
        "question_standardisee",
        "question_originale",
        "bonne_reponse"
    ]
].sort_values(
    ["parcours", "numero_question"]
).reset_index(drop=True)


# ============================================================
# 3. Création du modèle de correction des tests finaux
# ============================================================

df_correction_final_template = df_mapping_questions_finales.copy()

df_correction_final_template = df_correction_final_template.rename(columns={
    "ancienne_colonne": "question_originale",
    "nouvelle_colonne": "question_standardisee"
})

df_correction_final_template["type_test"] = "final"
df_correction_final_template["numero_question"] = (
    df_correction_final_template["question_standardisee"]
    .str.extract(r"Q(\d+)", expand=False)
    .astype(int)
)

df_correction_final_template["bonne_reponse"] = ""

df_correction_final_template = df_correction_final_template[
    [
        "type_test",
        "parcours",
        "feuille",
        "numero_question",
        "question_standardisee",
        "question_originale",
        "bonne_reponse"
    ]
].sort_values(
    ["parcours", "numero_question"]
).reset_index(drop=True)


# ============================================================
# 4. Export des modèles de correction
# ============================================================

PATH_CORRECTION_MI_TEMPLATE = OUTPUT_DIR / "template_correction_tests_intermediaires.csv"
PATH_CORRECTION_FINAL_TEMPLATE = OUTPUT_DIR / "template_correction_tests_finaux.csv"

df_correction_mi_template.to_csv(
    PATH_CORRECTION_MI_TEMPLATE,
    index=False,
    encoding="utf-8-sig"
)

df_correction_final_template.to_csv(
    PATH_CORRECTION_FINAL_TEMPLATE,
    index=False,
    encoding="utf-8-sig"
)


print("\nModèle de correction tests intermédiaires généré :")
print(PATH_CORRECTION_MI_TEMPLATE)

print("\nModèle de correction tests finaux généré :")
print(PATH_CORRECTION_FINAL_TEMPLATE)


print("\nAperçu correction tests intermédiaires :")
display(df_correction_mi_template.head(20))

print("\nAperçu correction tests finaux :")
display(df_correction_final_template.head(20))


print("\nContrôle du nombre de questions par parcours — intermédiaire :")
display(
    df_correction_mi_template
    .groupby("parcours")["question_standardisee"]
    .count()
)

print("\nContrôle du nombre de questions par parcours — final :")
display(
    df_correction_final_template
    .groupby("parcours")["question_standardisee"]
    .count()
)

Mapping intermédiaire existe : True
Mapping final existe : True

Mapping intermédiaire : (60, 4)
Mapping final : (60, 4)

Modèle de correction tests intermédiaires généré :
outputs/02_scoring_pedagogique/template_correction_tests_intermediaires.csv

Modèle de correction tests finaux généré :
outputs/02_scoring_pedagogique/template_correction_tests_finaux.csv

Aperçu correction tests intermédiaires :


,type_test,parcours,feuille,numero_question,question_standardisee,question_originale,bonne_reponse
0,intermediaire,BI,MI-TEST-BI,1,mi_Q01,Quel niveau d'analyse repond a la question : Q...,
1,intermediaire,BI,MI-TEST-BI,2,mi_Q02,Lequel de ces ensembles constitue correctement...,
2,intermediaire,BI,MI-TEST-BI,3,mi_Q03,Quel mode de connexion Power BI est recommande...,
3,intermediaire,BI,MI-TEST-BI,4,mi_Q04,Un analyste Elite Power BI doit principalement...,
4,intermediaire,BI,MI-TEST-BI,5,mi_Q05,"Dans Power Query, l'etape Load du modele ETL c...",
5,intermediaire,BI,MI-TEST-BI,6,mi_Q06,Le Fill Down dans Power Query permet de corrig...,
6,intermediaire,BI,MI-TEST-BI,7,mi_Q07,Quelle est la methode recommandee pour l'Unpiv...,
7,intermediaire,BI,MI-TEST-BI,8,mi_Q08,"Dans l'architecture 3 couches Power Query, que...",
8,intermediaire,BI,MI-TEST-BI,9,mi_Q09,Pourquoi faut-il imperativement typer la colon...,
9,intermediaire,BI,MI-TEST-BI,10,mi_Q10,Quel est l'ordre correct des operations lors d...,



Aperçu correction tests finaux :


,type_test,parcours,feuille,numero_question,question_standardisee,question_originale,bonne_reponse
0,final,BI,FINAL-TEST-BI,1,final_Q01,Une source contient des lignes Total et des en...,
1,final,BI,FINAL-TEST-BI,2,final_Q02,Un champ Matricule contient 00125. Quel type c...,
2,final,BI,FINAL-TEST-BI,3,final_Q03,"Une table a les colonnes Produit, Jan, Fev, Ma...",
3,final,BI,FINAL-TEST-BI,4,final_Q04,Une colonne Code contient MG-TNR-2026. Vous vo...,
4,final,BI,FINAL-TEST-BI,5,final_Q05,"Avant un Merge entre Ventes et Produits, quell...",
5,final,BI,FINAL-TEST-BI,6,final_Q06,Pourquoi construire un modele en etoile au lie...,
6,final,BI,FINAL-TEST-BI,7,final_Q07,"Apres avoir relie Dim_Produit a Fact_Ventes, q...",
7,final,BI,FINAL-TEST-BI,8,final_Q08,Vous devez calculer un taux de marge qui chang...,
8,final,BI,FINAL-TEST-BI,9,final_Q09,Quelle fonction DAX sert a recalculer une mesu...,
9,final,BI,FINAL-TEST-BI,10,final_Q10,"Vous classez les ventes en Faible, Moyen, Fort...",



Contrôle du nombre de questions par parcours — intermédiaire :


parcours
BI    15
DA    15
DS    15
IA    15
Name: question_standardisee, dtype: int64


Contrôle du nombre de questions par parcours — final :


parcours
BI    15
DA    15
DS    15
IA    15
Name: question_standardisee, dtype: int64

# Mapping intermédiaire : 60 lignes
# Mapping final         : 60 lignes

# Intermédiaire :
DA = 15 questions
BI = 15 questions
DS = 15 questions
IA = 15 questions

# Final :
DA = 15 questions
BI = 15 questions
DS = 15 questions
IA = 15 questions

# Cellule 9 bis — Aide pour voir les réponses disponibles

In [9]:
# ============================================================
# 2.8 bis. Aide au remplissage des barèmes
# Extraction des réponses observées par question
# ============================================================

def extraire_reponses_observees(df, colonnes_questions, type_test, parcours_col=None):
    """
    Extrait les réponses observées pour chaque question.
    Utile pour remplir les barèmes avec les valeurs exactes présentes dans les données.
    """
    lignes = []
    
    if parcours_col is None:
        for question in colonnes_questions:
            valeurs = (
                df[question]
                .dropna()
                .astype(str)
                .str.strip()
            )
            
            comptage = valeurs.value_counts(dropna=False)
            
            for reponse, effectif in comptage.items():
                lignes.append({
                    "type_test": type_test,
                    "parcours": "INITIAL",
                    "question": question,
                    "reponse_observee": reponse,
                    "effectif": int(effectif)
                })
    
    else:
        for parcours in sorted(df[parcours_col].dropna().unique()):
            df_parcours = df[df[parcours_col] == parcours]
            
            for question in colonnes_questions:
                valeurs = (
                    df_parcours[question]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )
                
                comptage = valeurs.value_counts(dropna=False)
                
                for reponse, effectif in comptage.items():
                    lignes.append({
                        "type_test": type_test,
                        "parcours": parcours,
                        "question": question,
                        "reponse_observee": reponse,
                        "effectif": int(effectif)
                    })
    
    return pd.DataFrame(lignes)


# Réponses observées dans le test initial
df_reponses_initial = extraire_reponses_observees(
    df_test_initial,
    colonnes_q_initial,
    type_test="initial",
    parcours_col=None
)

# Réponses observées dans les tests intermédiaires
df_reponses_mi = extraire_reponses_observees(
    df_tests_intermediaires,
    colonnes_q_mi,
    type_test="intermediaire",
    parcours_col="parcours_test_intermediaire"
)

# Réponses observées dans les tests finaux
df_reponses_final = extraire_reponses_observees(
    df_tests_finaux,
    colonnes_q_final,
    type_test="final",
    parcours_col="parcours_test_final"
)


# Export des fichiers d'aide
PATH_REPONSES_INITIAL = OUTPUT_DIR / "aide_reponses_observees_test_initial.csv"
PATH_REPONSES_MI = OUTPUT_DIR / "aide_reponses_observees_tests_intermediaires.csv"
PATH_REPONSES_FINAL = OUTPUT_DIR / "aide_reponses_observees_tests_finaux.csv"

df_reponses_initial.to_csv(PATH_REPONSES_INITIAL, index=False, encoding="utf-8-sig")
df_reponses_mi.to_csv(PATH_REPONSES_MI, index=False, encoding="utf-8-sig")
df_reponses_final.to_csv(PATH_REPONSES_FINAL, index=False, encoding="utf-8-sig")


print("Fichiers d'aide générés :")
print(PATH_REPONSES_INITIAL)
print(PATH_REPONSES_MI)
print(PATH_REPONSES_FINAL)

print("\nAperçu réponses observées — test initial :")
display(df_reponses_initial.head(30))

print("\nAperçu réponses observées — tests intermédiaires :")
display(df_reponses_mi.head(30))

print("\nAperçu réponses observées — tests finaux :")
display(df_reponses_final.head(30))

Fichiers d'aide générés :
outputs/02_scoring_pedagogique/aide_reponses_observees_test_initial.csv
outputs/02_scoring_pedagogique/aide_reponses_observees_tests_intermediaires.csv
outputs/02_scoring_pedagogique/aide_reponses_observees_tests_finaux.csv

Aperçu réponses observées — test initial :


,type_test,parcours,question,reponse_observee,effectif
0,initial,INITIAL,Q1 - Resume ventes Excel,B. Tableau croise dynamique (TCD),110
1,initial,INITIAL,Q1 - Resume ventes Excel,C. Formule SOMME(),67
2,initial,INITIAL,Q1 - Resume ventes Excel,E. Je ne sais pas,55
3,initial,INITIAL,Q1 - Resume ventes Excel,A. Filtre automatique,41
4,initial,INITIAL,Q1 - Resume ventes Excel,D. Graphique en barres,32
5,initial,INITIAL,Q2 - Import CSV Excel,E. Je ne sais pas,130
6,initial,INITIAL,Q2 - Import CSV Excel,D. Power Query,57
7,initial,INITIAL,Q2 - Import CSV Excel,B. Copier-coller manuellement,49
8,initial,INITIAL,Q2 - Import CSV Excel,C. Formules SI imbriquees,42
9,initial,INITIAL,Q2 - Import CSV Excel,A. Macros VBA manuelles,27



Aperçu réponses observées — tests intermédiaires :


,type_test,parcours,question,reponse_observee,effectif
0,intermediaire,BI,mi_Q01,C) Descriptif,41
1,intermediaire,BI,mi_Q01,D) Diagnostique,10
2,intermediaire,BI,mi_Q01,B) Prescriptif,5
3,intermediaire,BI,mi_Q01,A) Predictif,4
4,intermediaire,BI,mi_Q02,"B) Power BI Desktop, Service, Mobile et Data G...",48
5,intermediaire,BI,mi_Q02,A) Power Query uniquement,10
6,intermediaire,BI,mi_Q02,"C) Excel, Access et Tableau",2
7,intermediaire,BI,mi_Q03,C) Import — donnees copiees dans Power BI,45
8,intermediaire,BI,mi_Q03,D) Push Dataset — envoi de donnees en temps reel,8
9,intermediaire,BI,mi_Q03,A) DirectQuery — lecture en direct sur la source,6



Aperçu réponses observées — tests finaux :


,type_test,parcours,question,reponse_observee,effectif
0,final,BI,final_Q01,B) Des mesures fausses car ces lignes peuvent ...,42
1,final,BI,final_Q01,D) Un doublon uniquement visible dans la vue D...,12
2,final,BI,final_Q01,A) Aucun risque si les visuels utilisent des f...,2
3,final,BI,final_Q01,C) Un ralentissement sans impact sur les chiffres,1
4,final,BI,final_Q02,B) Texte,35
5,final,BI,final_Q02,A) Nombre entier avec format 00000,13
6,final,BI,final_Q02,D) Nombre decimal,8
7,final,BI,final_Q02,C) Nombre entier puis conversion dans DAX,3
8,final,BI,final_Q03,"C) Depivoter Jan, Fev, Mar en Mois/Valeur",48
9,final,BI,final_Q03,A) Dupliquer une requete par mois,5


# Méthode manuelle pour remplir bonne_reponse
1. Ouvrir le fichier d’aide

## Ouvre d’abord :
aide_reponses_observees_test_initial.csv
aide_reponses_observees_tests_intermediaires.csv
aide_reponses_observees_tests_finaux.csv


Oui, mais il faut distinguer deux choses :

**1. Cellule qui remplit `bonne_reponse`**
Elle remplit seulement le **barème correct**.
Elle ne calcule pas encore la note de chaque apprenant.

**2. Cellules suivantes de scoring**
Elles vont comparer :

```text
réponse de l’apprenant == bonne_reponse
```

et calculer la note de chaque apprenant.

## Barème

### Test initial

```text
20 questions au total
DA : 5 questions
BI : 5 questions
DS : 5 questions
IA : 5 questions

Note totale : /20
Note par domaine : /5
```

### Tests intermédiaires

```text
15 questions par parcours
DA : 15 questions
BI : 15 questions
DS : 15 questions
IA : 15 questions

Chaque apprenant répond seulement au test de son parcours.
Note intermédiaire : /15
```

Donc le fichier barème contient **60 lignes** parce que :

```text
4 parcours × 15 questions = 60
```

### Tests finaux

Même logique :

```text
15 questions par parcours
DA : 15 questions
BI : 15 questions
DS : 15 questions
IA : 15 questions

Chaque apprenant répond seulement au test final de son parcours.
Note finale : /15
```

Donc :

```text
4 parcours × 15 questions = 60
```

Résumé :

```text
Test initial : /20
Test intermédiaire : /15
Test final : /15
```


# Cellule 9 ter — Remplissage automatique du barème proposé

In [10]:
# ============================================================
# 2.8 ter. Remplissage automatique des 140 bonnes réponses
# Barème proposé à valider pédagogiquement
# ============================================================

# -----------------------------
# 1. Barème test initial /20
# -----------------------------

bonnes_reponses_initial = {
    1: "B. Tableau croise dynamique (TCD)",
    2: "D. Power Query",
    3: "C. Power Pivot avec relations",
    4: "D. Graphique 3D explose",
    5: "B. GRANDE.VALEUR(plage;2)",
    6: "C. Mesure=visualisation Colonne=stockee",
    7: "B. CALCULATE(SUM SAMEPERIODLASTYEAR)",
    8: "D. Definir relations entre tables",
    9: "B. Row-Level Security (RLS)",
    10: "D. Radar ou tableau matriciel conditionnel",
    11: "C. pandas",
    12: "D. Overfitting",
    13: "B. K-Means Clustering",
    14: "C. FastAPI ou Flask + joblib ou pickle",
    15: "C. Imputer avec mediane ou modele",
    16: "D. Definir role et contraintes du modele",
    17: "B. RAG - indexer et recuperer dynamiquement",
    18: "D. Transformer texte en vecteurs numeriques",
    19: "C. Raisonnement LLM (function calling)",
    20: "D. JSON structure avec schema defini"
}

# -----------------------------
# 2. Barème tests intermédiaires /15 par parcours
# -----------------------------

bonnes_reponses_mi = {
    ("DA", "mi_Q01"): "C) Garbage In, Garbage Out — des donnees sources de mauvaise qualite produisent des analyses de mauvaise qualite",
    ("DA", "mi_Q02"): "D) Exactitude — la valeur ne represente pas correctement la realite",
    ("DA", "mi_Q03"): "B) Nettoyer, filtrer, enrichir et formater les donnees",
    ("DA", "mi_Q04"): "B) Cliquer sur Transformer les donnees pour ouvrir l'editeur",
    ("DA", "mi_Q05"): "C) Il utilise l'evaluation paresseuse (Lazy Evaluation) — l'apercu est partiel mais les transformations s'appliquent a toutes les lignes lors du chargement",
    ("DA", "mi_Q06"): "B) Dupliquer cree une copie independante (source interrogee 2 fois), Referencer herite les modifications de l'originale (source interrogee 1 seule fois)",
    ("DA", "mi_Q07"): "B) Table.SelectColumns liste uniquement les colonnes a garder — si la source ajoute de nouvelles colonnes, elles sont automatiquement ignorees (requete resistante)",
    ("DA", "mi_Q08"): "C) Le zero initial est perdu — la valeur devient 6000, ce qui casse toutes les jointures geographiques",
    ("DA", "mi_Q09"): "C) Connecter > Nommer > Filtrer > Supprimer colonnes inutiles > Nettoyer > Typer > Charger",
    ("DA", "mi_Q10"): "C) En toute derniere etape de la requete, apres toutes les transformations",
    ("DA", "mi_Q11"): "C) Au centre, reliee a toutes les tables de dimensions par des cles etrangeres",
    ("DA", "mi_Q12"): "B) Elle doit etre strictement unique — aucun doublon ni valeur vide n'est tolere",
    ("DA", "mi_Q13"): "C) La ligne DIR n'a aucune correspondance — ses faits associes sont exclus des calculs (lignes orphelines)",
    ("DA", "mi_Q14"): "C) Left Outer Join — toutes les lignes de la table Ventes",
    ("DA", "mi_Q15"): "B) Parce que la mega-table duplique les donnees de dimension ligne par ligne, alourdit la memoire et ralentit les calculs du moteur xVelocity",

    ("BI", "mi_Q01"): "C) Descriptif",
    ("BI", "mi_Q02"): "B) Power BI Desktop, Service, Mobile et Data Gateway",
    ("BI", "mi_Q03"): "C) Import — donnees copiees dans Power BI",
    ("BI", "mi_Q04"): "C) Pourquoi cela s'est-il passe — et que faut-il faire ?",
    ("BI", "mi_Q05"): "C) L'envoi des donnees propres vers le modele Power BI",
    ("BI", "mi_Q06"): "B) Les cellules fusionnees Excel qui deviennent null a l'import",
    ("BI", "mi_Q07"): "C) Selectionner les colonnes stables puis Depivoter les autres colonnes",
    ("BI", "mi_Q08"): "C) Dim_ / Fact_ — tables finales (Load active)",
    ("BI", "mi_Q09"): "B) Pour eviter la perte des zeros initiaux qui brise toutes les jointures",
    ("BI", "mi_Q10"): "B) Fill Down puis Supprimer les lignes vides",
    ("BI", "mi_Q11"): "C) 1 a plusieurs (1:*)",
    ("BI", "mi_Q12"): "B) Pour permettre l'analyse Annee/Trimestre/Mois et connecter plusieurs tables de faits sur une seule dimension temps",
    ("BI", "mi_Q13"): "B) Il reduit la duplication des donnees et optimise la compression VertiPaq pour des calculs plus rapides",
    ("BI", "mi_Q14"): "B) La cardinalite bascule automatiquement en plusieurs-a-plusieurs (*:*)",
    ("BI", "mi_Q15"): "C) Creer une table de liaison (Bridge table) contenant les cles des deux tables",

    ("DS", "mi_Q01"): "C) Predictive",
    ("DS", "mi_Q02"): "B) Pour isoler les dependances et eviter les conflits entre projets",
    ("DS", "mi_Q03"): "C) Dictionnaire {cle: valeur}",
    ("DS", "mi_Q04"): "C) Donnees non structurees (images, texte libre, audio)",
    ("DS", "mi_Q05"): "C) L'extraction depuis la source (fichier, base de donnees, API)",
    ("DS", "mi_Q06"): "A) df.describe()",
    ("DS", "mi_Q07"): "C) pd.merge()",
    ("DS", "mi_Q08"): "D) Boxplot",
    ("DS", "mi_Q09"): "B) (x - min) / (max - min)",
    ("DS", "mi_Q10"): "C) Il existe une forte liaison lineaire positive, mais pas necessairement une causalite",
    ("DS", "mi_Q11"): "B) Le supervise utilise des donnees etiquetees (variable cible connue), le non supervise travaille sans etiquettes",
    ("DS", "mi_Q12"): "B) Overfitting — le modele a memorise les donnees d'entrainement",
    ("DS", "mi_Q13"): "C) 70-80 % entrainement / 20-30 % test",
    ("DS", "mi_Q14"): "C) SMOTE — generer synthetiquement de nouveaux exemples de la classe minoritaire",
    ("DS", "mi_Q15"): "B) Car il combine plusieurs arbres entraines sur des sous-ensembles aleatoires, reduisant la variance",

    ("IA", "mi_Q01"): "C) Parce qu'il predit le token suivant le plus probable — il n'a aucun mecanisme interne de verification factuelle",
    ("IA", "mi_Q02"): "C) Le Lost in the Middle — les LLMs retiennent mieux les informations en debut et en fin de contexte qu'en milieu",
    ("IA", "mi_Q03"): "C) Chain-of-Thought (CoT) — Reflechis etape par etape avant de repondre",
    ("IA", "mi_Q04"): "D) temperature = 0 — le modele choisit toujours le token le plus probable, resultats stables",
    ("IA", "mi_Q05"): "D) Function Calling / Tool Use — l'API force le modele a retourner un objet structure conforme au schema fourni",
    ("IA", "mi_Q06"): "D) Garde-fous et interdictions — les comportements interdits definis explicitement",
    ("IA", "mi_Q07"): "B) 200 a 500 mots — assez detaille pour etre precis, assez court pour ne pas diluer les instructions",
    ("IA", "mi_Q08"): "B) Le System Prompt est charge automatiquement a chaque conversation ; le Skill n'est charge que lorsqu'on l'active avec $skill-id dans le chat",
    ("IA", "mi_Q09"): "C) Les procedures detaillees de calcul de bulletin de paie — utilisees 2 fois par semaine seulement",
    ("IA", "mi_Q10"): "C) Le Tool Python permet au LLM d'obtenir des donnees reelles du monde (date du jour, appel API, calcul precis) que sa connaissance interne ne peut pas fournir",
    ("IA", "mi_Q11"): "C) Model Context Protocol — standard universel permettant a un LLM de se connecter a des outils et services externes (bases de donnees, APIs, fichiers)",
    ("IA", "mi_Q12"): "B) L'environnement de developpement IA (Claude Code + serveurs MCP configures) qui permet de creer, tester et executer des pipelines en langage naturel dans le terminal",
    ("IA", "mi_Q13"): "C) Via le MCP Server N8N (mcp-n8n) — Antigravity envoie des commandes en langage naturel, traduites en actions sur les noeuds N8N par le serveur MCP",
    ("IA", "mi_Q14"): "C) RAG — decouper le PDF en chunks, vectoriser chaque chunk (embedding), et ne recuperer que les passages pertinents lors de chaque question",
    ("IA", "mi_Q15"): "C) A transformer chaque chunk de texte en vecteur numerique permettant une recherche par similarite semantique — deux phrases proches conceptuellement auront des vecteurs proches"
}

# -----------------------------
# 3. Barème tests finaux /15 par parcours
# -----------------------------

bonnes_reponses_final = {
    ("DA", "final_Q01"): "D) Changer le type de la colonne en Date avec la locale adaptee",
    ("DA", "final_Q02"): "B) Remplacer les valeurs null par A traiter dans la colonne Statut",
    ("DA", "final_Q03"): "D) Fusionner sur CodeClient puis developper Region",
    ("DA", "final_Q04"): "B) Colonne conditionnelle",
    ("DA", "final_Q05"): "B) Fact_Factures",
    ("DA", "final_Q06"): "D) Agences 1 vers Transactions plusieurs",
    ("DA", "final_Q07"): "A) Une colonne calculee",
    ("DA", "final_Q08"): "A) Nb = COUNTROWS(Reclamations)",
    ("DA", "final_Q09"): "D) DIVIDE([Admis], [Inscrits])",
    ("DA", "final_Q10"): "C) CALCULATE([Total Montant], Filiere[Nom] = \"Informatique\")",
    ("DA", "final_Q11"): "C) Courbe",
    ("DA", "final_Q12"): "A) Segment connecte aux TCD",
    ("DA", "final_Q13"): "A) Barres groupees",
    ("DA", "final_Q14"): "D) Definir les questions metier et les indicateurs",
    ("DA", "final_Q15"): "A) Segmenter par groupe, periode et module avant de conclure",

    ("BI", "final_Q01"): "B) Des mesures fausses car ces lignes peuvent etre agregees",
    ("BI", "final_Q02"): "B) Texte",
    ("BI", "final_Q03"): "C) Depivoter Jan, Fev, Mar en Mois/Valeur",
    ("BI", "final_Q04"): "C) Extraire le texte entre delimitateurs",
    ("BI", "final_Q05"): "B) Verifier cle commune, type et espaces inutiles",
    ("BI", "final_Q06"): "B) Pour eviter la duplication, clarifier les relations et fiabiliser les mesures",
    ("BI", "final_Q07"): "D) Cardinalite, direction de filtre et unicite cote dimension",
    ("BI", "final_Q08"): "B) Mesure",
    ("BI", "final_Q09"): "A) CALCULATE",
    ("BI", "final_Q10"): "D) SWITCH(TRUE(), ...)",
    ("BI", "final_Q11"): "A) Faire ressortir tendances, ecarts et priorites",
    ("BI", "final_Q12"): "D) Pour garantir que les chiffres restent coherents quand l'utilisateur explore",
    ("BI", "final_Q13"): "C) L'utilisateur peut confondre les categories et mal interpreter",
    ("BI", "final_Q14"): "C) Barres groupees",
    ("BI", "final_Q15"): "A) Besoins/KPI, modele fiable, choix des visuels, mise en page, test de lisibilite",

    ("DS", "final_Q01"): "A) Comprendre les variables et transformer les donnees en information exploitable",
    ("DS", "final_Q02"): "D) Pour stabiliser les proprietes statistiques du signal",
    ("DS", "final_Q03"): "D) Donnees tabulaires structurees",
    ("DS", "final_Q04"): "B) df.describe()",
    ("DS", "final_Q05"): "D) Boxplot",
    ("DS", "final_Q06"): "A) R2 ajuste",
    ("DS", "final_Q07"): "A) Reduire le bruit et aider le modele a generaliser",
    ("DS", "final_Q08"): "B) Des groupes assez coherents et separes",
    ("DS", "final_Q09"): "D) t-SNE",
    ("DS", "final_Q10"): "C) Non supervise",
    ("DS", "final_Q11"): "B) stratify",
    ("DS", "final_Q12"): "B) Tester plusieurs hyperparametres et choisir une configuration performante",
    ("DS", "final_Q13"): "D) Khi-deux ou Fisher selon les effectifs",
    ("DS", "final_Q14"): "B) Random Forest",
    ("DS", "final_Q15"): "C) Objectif metier, donnees disponibles et preparation necessaire",

    ("IA", "final_Q01"): "B) Extraire le fichier, nettoyer/transformer les donnees, charger le resultat exploitable dans le stockage cible",
    ("IA", "final_Q02"): "A) Les mesures et cles vers les dimensions",
    ("IA", "final_Q03"): "B) Workspace / Espace de travail > Skills",
    ("IA", "final_Q04"): "A) Administration > Reglages > Integrations",
    ("IA", "final_Q05"): "C) Administration > Fonctions > Nouvelle fonction",
    ("IA", "final_Q06"): "D) Valider et structurer les donnees recues et renvoyees",
    ("IA", "final_Q07"): "D) Generer des graphiques interactifs ou exportables",
    ("IA", "final_Q08"): "C) Stocker et exposer par API les contenus/rapports/sessions",
    ("IA", "final_Q09"): "D) Titre, contenu MDX, metriques, secteur, statut du rapport",
    ("IA", "final_Q10"): "C) Afficher les rapports et declencher/recuperer les resultats du pipeline cote application",
    ("IA", "final_Q11"): "C) Des nodes connectes qui executent des actions successives",
    ("IA", "final_Q12"): "B) Lancer l'ETL, appeler les API, publier dans Directus et notifier",
    ("IA", "final_Q13"): "A) Piloter ou modifier des workflows et integrations par instructions/outils sans tout faire manuellement dans l'interface",
    ("IA", "final_Q14"): "D) Verifier l'endpoint/API appele, le slug/id et le mapping des champs",
    ("IA", "final_Q15"): "D) Administration > Reglages > Connexion"
}

# -----------------------------
# 4. Chargement des fichiers de barème
# -----------------------------

PATH_CORRECTION_INITIAL = OUTPUT_DIR / "template_correction_test_initial.csv"
PATH_CORRECTION_MI = OUTPUT_DIR / "template_correction_tests_intermediaires.csv"
PATH_CORRECTION_FINAL = OUTPUT_DIR / "template_correction_tests_finaux.csv"

df_correction_initiale = pd.read_csv(PATH_CORRECTION_INITIAL)
df_correction_mi = pd.read_csv(PATH_CORRECTION_MI)
df_correction_final = pd.read_csv(PATH_CORRECTION_FINAL)

# -----------------------------
# 5. Remplissage automatique
# -----------------------------

df_correction_initiale["bonne_reponse"] = df_correction_initiale["numero_question"].map(
    bonnes_reponses_initial
)

df_correction_mi["bonne_reponse"] = df_correction_mi.apply(
    lambda row: bonnes_reponses_mi.get(
        (str(row["parcours"]).strip(), str(row["question_standardisee"]).strip()),
        np.nan
    ),
    axis=1
)

df_correction_final["bonne_reponse"] = df_correction_final.apply(
    lambda row: bonnes_reponses_final.get(
        (str(row["parcours"]).strip(), str(row["question_standardisee"]).strip()),
        np.nan
    ),
    axis=1
)

# -----------------------------
# 6. Vérification
# -----------------------------

print("Vides test initial :", df_correction_initiale["bonne_reponse"].isna().sum())
print("Vides tests intermédiaires :", df_correction_mi["bonne_reponse"].isna().sum())
print("Vides tests finaux :", df_correction_final["bonne_reponse"].isna().sum())

# -----------------------------
# 7. Export des barèmes complétés
# -----------------------------

df_correction_initiale.to_csv(PATH_CORRECTION_INITIAL, index=False, encoding="utf-8-sig")
df_correction_mi.to_csv(PATH_CORRECTION_MI, index=False, encoding="utf-8-sig")
df_correction_final.to_csv(PATH_CORRECTION_FINAL, index=False, encoding="utf-8-sig")

print("Barèmes complétés et exportés.")
print("Total bonnes réponses :", len(df_correction_initiale) + len(df_correction_mi) + len(df_correction_final))

Vides test initial : 0
Vides tests intermédiaires : 0
Vides tests finaux : 0
Barèmes complétés et exportés.
Total bonnes réponses : 140


# Cellule 9 — Chargement et vérification des barèmes complétés

In [11]:
# ============================================================
# 2.8. Chargement et vérification des barèmes complétés
# ============================================================

PATH_CORRECTION_INITIAL = OUTPUT_DIR / "template_correction_test_initial.csv"
PATH_CORRECTION_MI = OUTPUT_DIR / "template_correction_tests_intermediaires.csv"
PATH_CORRECTION_FINAL = OUTPUT_DIR / "template_correction_tests_finaux.csv"

print("Fichier correction initiale existe :", PATH_CORRECTION_INITIAL.exists())
print("Fichier correction intermédiaire existe :", PATH_CORRECTION_MI.exists())
print("Fichier correction finale existe :", PATH_CORRECTION_FINAL.exists())

if not PATH_CORRECTION_INITIAL.exists():
    raise FileNotFoundError("Le fichier de correction du test initial est introuvable.")

if not PATH_CORRECTION_MI.exists():
    raise FileNotFoundError("Le fichier de correction des tests intermédiaires est introuvable.")

if not PATH_CORRECTION_FINAL.exists():
    raise FileNotFoundError("Le fichier de correction des tests finaux est introuvable.")


# Chargement des fichiers de correction
df_correction_initiale = pd.read_csv(PATH_CORRECTION_INITIAL)
df_correction_mi = pd.read_csv(PATH_CORRECTION_MI)
df_correction_final = pd.read_csv(PATH_CORRECTION_FINAL)


def est_vide(valeur):
    """
    Vérifie si une cellule de bonne réponse est vide.
    """
    if pd.isna(valeur):
        return True
    
    valeur = str(valeur).strip()
    
    return valeur == "" or valeur.lower() in ["nan", "none", "null", "nat"]


def verifier_bareme(df, nom_bareme, nb_lignes_attendu):
    """
    Vérifie qu'un barème est complet :
    - nombre de lignes attendu ;
    - présence de la colonne bonne_reponse ;
    - absence de bonnes réponses vides.
    """
    print(f"\n===== Vérification : {nom_bareme} =====")
    
    print("Dimensions :", df.shape)
    
    if df.shape[0] != nb_lignes_attendu:
        print(f"Attention : {df.shape[0]} lignes trouvées au lieu de {nb_lignes_attendu}.")
    else:
        print("Nombre de lignes : OK")
    
    if "bonne_reponse" not in df.columns:
        raise ValueError(f"Colonne bonne_reponse absente dans {nom_bareme}.")
    
    nb_vides = df["bonne_reponse"].apply(est_vide).sum()
    nb_remplies = df.shape[0] - nb_vides
    
    print("Bonnes réponses remplies :", nb_remplies)
    print("Bonnes réponses vides :", nb_vides)
    
    if nb_vides > 0:
        print("\nLignes encore non complétées :")
        display(df[df["bonne_reponse"].apply(est_vide)].head(20))
    else:
        print("Barème complet : OK")
    
    return nb_vides


nb_vides_initial = verifier_bareme(
    df_correction_initiale,
    "Barème test initial",
    20
)

nb_vides_mi = verifier_bareme(
    df_correction_mi,
    "Barème tests intermédiaires",
    60
)

nb_vides_final = verifier_bareme(
    df_correction_final,
    "Barème tests finaux",
    60
)


total_vides = nb_vides_initial + nb_vides_mi + nb_vides_final

print("\n===== SYNTHÈSE DES BARÈMES =====")
print("Réponses manquantes test initial :", nb_vides_initial)
print("Réponses manquantes tests intermédiaires :", nb_vides_mi)
print("Réponses manquantes tests finaux :", nb_vides_final)
print("Total réponses manquantes :", total_vides)

if total_vides > 0:
    raise ValueError(
        "Les barèmes ne sont pas encore complets. "
        "Complète toutes les colonnes bonne_reponse avant de continuer."
    )
else:
    print("Tous les barèmes sont complets. On peut passer au calcul des scores.")

Fichier correction initiale existe : True
Fichier correction intermédiaire existe : True
Fichier correction finale existe : True

===== Vérification : Barème test initial =====
Dimensions : (20, 4)
Nombre de lignes : OK
Bonnes réponses remplies : 20
Bonnes réponses vides : 0
Barème complet : OK

===== Vérification : Barème tests intermédiaires =====
Dimensions : (60, 7)
Nombre de lignes : OK
Bonnes réponses remplies : 60
Bonnes réponses vides : 0
Barème complet : OK

===== Vérification : Barème tests finaux =====
Dimensions : (60, 7)
Nombre de lignes : OK
Bonnes réponses remplies : 60
Bonnes réponses vides : 0
Barème complet : OK

===== SYNTHÈSE DES BARÈMES =====
Réponses manquantes test initial : 0
Réponses manquantes tests intermédiaires : 0
Réponses manquantes tests finaux : 0
Total réponses manquantes : 0
Tous les barèmes sont complets. On peut passer au calcul des scores.


# Cellule 10 — Calcul des scores du test initial

In [12]:
# ============================================================
# 2.9. Calcul des scores du test initial /20
# ============================================================

df_scores_initial = df_test_initial[["IDENTIFICATION"]].copy()

# Sécurité : vérifier les colonnes
questions_absentes = [
    q for q in df_correction_initiale["question"]
    if q not in df_test_initial.columns
]

if len(questions_absentes) > 0:
    raise ValueError(f"Questions absentes dans df_test_initial : {questions_absentes}")

# Calcul question par question
for _, row in df_correction_initiale.iterrows():
    numero = int(row["numero_question"])
    domaine = row["domaine"]
    question = row["question"]
    bonne_reponse = row["bonne_reponse"]
    
    col_score = f"init_score_Q{numero:02d}"
    col_inconnue = f"init_unknown_Q{numero:02d}"
    
    df_scores_initial[col_score] = df_test_initial[question].apply(
        lambda rep: corriger_reponse(rep, bonne_reponse)
    )
    
    df_scores_initial[col_inconnue] = df_test_initial[question].apply(
        est_reponse_inconnue
    )

# Scores par domaine /5
domaines_initial = {
    "DA": [f"init_score_Q{i:02d}" for i in range(1, 6)],
    "BI": [f"init_score_Q{i:02d}" for i in range(6, 11)],
    "DS": [f"init_score_Q{i:02d}" for i in range(11, 16)],
    "IA": [f"init_score_Q{i:02d}" for i in range(16, 21)]
}

for domaine, colonnes in domaines_initial.items():
    df_scores_initial[f"score_initial_{domaine}"] = df_scores_initial[colonnes].sum(axis=1)
    df_scores_initial[f"score_initial_{domaine}_pct"] = (
        df_scores_initial[f"score_initial_{domaine}"] / 5 * 100
    )
    df_scores_initial[f"niveau_initial_{domaine}"] = df_scores_initial[
        f"score_initial_{domaine}_pct"
    ].apply(attribuer_niveau_depuis_pourcentage)

# Score global /20
colonnes_scores_questions = [f"init_score_Q{i:02d}" for i in range(1, 21)]

df_scores_initial["score_initial_total"] = df_scores_initial[colonnes_scores_questions].sum(axis=1)
df_scores_initial["score_initial_pourcentage"] = df_scores_initial["score_initial_total"] / 20 * 100
df_scores_initial["niveau_initial_global"] = df_scores_initial[
    "score_initial_pourcentage"
].apply(attribuer_niveau_depuis_pourcentage)

# Nombre de réponses inconnues
colonnes_unknown = [f"init_unknown_Q{i:02d}" for i in range(1, 21)]
df_scores_initial["unknown_total_initial"] = df_scores_initial[colonnes_unknown].sum(axis=1)

# Export
PATH_SCORES_INITIAL = OUTPUT_DIR / "01_scores_test_initial.csv"

df_scores_initial.to_csv(
    PATH_SCORES_INITIAL,
    index=False,
    encoding="utf-8-sig"
)

print("Scores du test initial calculés.")
print("Fichier généré :", PATH_SCORES_INITIAL)
print("Dimensions :", df_scores_initial.shape)

print("\nRésumé score initial total /20 :")
display(df_scores_initial["score_initial_total"].describe())

print("\nRépartition niveau initial global :")
display(df_scores_initial["niveau_initial_global"].value_counts())

display(df_scores_initial.head())

Scores du test initial calculés.
Fichier généré : outputs/02_scoring_pedagogique/01_scores_test_initial.csv
Dimensions : (305, 57)

Résumé score initial total /20 :


count    305.000000
mean       4.577049
std        4.412318
min        0.000000
25%        1.000000
50%        4.000000
75%        6.000000
max       20.000000
Name: score_initial_total, dtype: float64


Répartition niveau initial global :


niveau_initial_global
Débutant         268
Intermédiaire     22
Avancé            15
Name: count, dtype: int64

,IDENTIFICATION,init_score_Q01,init_unknown_Q01,init_score_Q02,init_unknown_Q02,init_score_Q03,init_unknown_Q03,init_score_Q04,init_unknown_Q04,init_score_Q05,init_unknown_Q05,init_score_Q06,init_unknown_Q06,init_score_Q07,init_unknown_Q07,init_score_Q08,init_unknown_Q08,init_score_Q09,init_unknown_Q09,init_score_Q10,init_unknown_Q10,init_score_Q11,init_unknown_Q11,init_score_Q12,init_unknown_Q12,init_score_Q13,init_unknown_Q13,init_score_Q14,init_unknown_Q14,init_score_Q15,init_unknown_Q15,init_score_Q16,init_unknown_Q16,init_score_Q17,init_unknown_Q17,init_score_Q18,init_unknown_Q18,init_score_Q19,init_unknown_Q19,init_score_Q20,init_unknown_Q20,score_initial_DA,score_initial_DA_pct,niveau_initial_DA,score_initial_BI,score_initial_BI_pct,niveau_initial_BI,score_initial_DS,score_initial_DS_pct,niveau_initial_DS,score_initial_IA,score_initial_IA_pct,niveau_initial_IA,score_initial_total,score_initial_pourcentage,niveau_initial_global,unknown_total_initial
0,DIEG2026-001,0,False,0,False,0,False,0,False,0,False,1,False,1,False,1,False,0,False,1,False,0,True,0,False,0,False,1,False,0,False,0,True,1,False,0,False,0,False,0,False,0,0.0,Débutant,4,80.0,Avancé,1,20.0,Débutant,1,20.0,Débutant,6,30.0,Débutant,2
1,DIEG2026-002,0,False,0,False,1,False,0,False,0,False,1,False,0,True,0,False,1,False,0,False,0,False,0,False,0,False,0,False,0,False,0,True,1,False,0,False,0,True,1,False,1,20.0,Débutant,2,40.0,Débutant,0,0.0,Débutant,2,40.0,Débutant,5,25.0,Débutant,3
2,DIEG2026-004,1,False,1,False,1,False,1,False,0,False,1,False,1,False,1,False,0,False,0,False,0,False,1,False,0,False,1,False,0,False,1,False,0,False,0,True,0,True,0,True,4,80.0,Avancé,3,60.0,Intermédiaire,2,40.0,Débutant,1,20.0,Débutant,10,50.0,Intermédiaire,3
3,DIEG2026-006,0,False,0,True,0,True,0,True,0,False,0,False,0,False,0,False,0,False,0,True,0,False,0,False,0,False,0,False,0,False,0,False,1,False,0,False,0,True,0,False,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,1,20.0,Débutant,1,5.0,Débutant,5
4,DIEG2026-005,0,False,1,False,1,False,1,False,0,True,1,False,1,False,0,False,1,False,0,False,0,True,0,True,1,False,0,False,1,False,1,False,1,False,0,False,1,False,1,False,3,60.0,Intermédiaire,3,60.0,Intermédiaire,2,40.0,Débutant,4,80.0,Avancé,12,60.0,Intermédiaire,3


# Cellule 11 — Calcul des scores des tests intermédiaires /15.

In [13]:
# ============================================================
# 2.10. Calcul des scores des tests intermédiaires /15
# ============================================================

df_scores_intermediaires = df_tests_intermediaires[
    ["IDENTIFICATION", "parcours_test_intermediaire", "feuille_source_intermediaire"]
].copy()

# Mapping du barème intermédiaire
bareme_mi = (
    df_correction_mi
    .set_index(["parcours", "question_standardisee"])["bonne_reponse"]
    .to_dict()
)

# Calcul question par question
for q in colonnes_q_mi:
    col_score = f"score_{q}"
    col_unknown = f"unknown_{q}"
    
    df_scores_intermediaires[col_score] = df_tests_intermediaires.apply(
        lambda row: corriger_reponse(
            row[q],
            bareme_mi.get((row["parcours_test_intermediaire"], q))
        ),
        axis=1
    )
    
    df_scores_intermediaires[col_unknown] = df_tests_intermediaires[q].apply(
        est_reponse_inconnue
    )

# Score total /15
colonnes_scores_mi = [f"score_{q}" for q in colonnes_q_mi]
colonnes_unknown_mi = [f"unknown_{q}" for q in colonnes_q_mi]

df_scores_intermediaires["score_intermediaire_total"] = df_scores_intermediaires[colonnes_scores_mi].sum(axis=1)
df_scores_intermediaires["score_intermediaire_pourcentage"] = (
    df_scores_intermediaires["score_intermediaire_total"] / 15 * 100
)

df_scores_intermediaires["niveau_intermediaire"] = df_scores_intermediaires[
    "score_intermediaire_pourcentage"
].apply(attribuer_niveau_depuis_pourcentage)

df_scores_intermediaires["unknown_total_intermediaire"] = df_scores_intermediaires[colonnes_unknown_mi].sum(axis=1)

# Export
PATH_SCORES_MI = OUTPUT_DIR / "02_scores_tests_intermediaires.csv"

df_scores_intermediaires.to_csv(
    PATH_SCORES_MI,
    index=False,
    encoding="utf-8-sig"
)

print("Scores des tests intermédiaires calculés.")
print("Fichier généré :", PATH_SCORES_MI)
print("Dimensions :", df_scores_intermediaires.shape)

print("\nRésumé score intermédiaire total /15 :")
display(df_scores_intermediaires["score_intermediaire_total"].describe())

print("\nRépartition des niveaux intermédiaires :")
display(df_scores_intermediaires["niveau_intermediaire"].value_counts())

print("\nRépartition par parcours :")
display(
    df_scores_intermediaires
    .groupby("parcours_test_intermediaire")["score_intermediaire_total"]
    .describe()
)

display(df_scores_intermediaires.head())

Scores des tests intermédiaires calculés.
Fichier généré : outputs/02_scoring_pedagogique/02_scores_tests_intermediaires.csv
Dimensions : (196, 37)

Résumé score intermédiaire total /15 :


count    196.000000
mean      10.811224
std        3.232856
min        2.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       15.000000
Name: score_intermediaire_total, dtype: float64


Répartition des niveaux intermédiaires :


niveau_intermediaire
Avancé           108
Intermédiaire     54
Débutant          34
Name: count, dtype: int64


Répartition par parcours :


,count,mean,std,min,25%,50%,75%,max
parcours_test_intermediaire,,,,,,,,
BI,60.0,9.933333,3.583704,2.0,7.0,10.0,13.00,15.0
DA,32.0,11.500000,3.100468,4.0,9.0,12.0,14.25,15.0
DS,44.0,11.795455,2.937960,4.0,10.0,12.5,14.25,15.0
IA,60.0,10.600000,2.923893,3.0,9.0,11.0,13.00,15.0


,IDENTIFICATION,parcours_test_intermediaire,feuille_source_intermediaire,score_mi_Q01,unknown_mi_Q01,score_mi_Q02,unknown_mi_Q02,score_mi_Q03,unknown_mi_Q03,score_mi_Q04,unknown_mi_Q04,score_mi_Q05,unknown_mi_Q05,score_mi_Q06,unknown_mi_Q06,score_mi_Q07,unknown_mi_Q07,score_mi_Q08,unknown_mi_Q08,score_mi_Q09,unknown_mi_Q09,score_mi_Q10,unknown_mi_Q10,score_mi_Q11,unknown_mi_Q11,score_mi_Q12,unknown_mi_Q12,score_mi_Q13,unknown_mi_Q13,score_mi_Q14,unknown_mi_Q14,score_mi_Q15,unknown_mi_Q15,score_intermediaire_total,score_intermediaire_pourcentage,niveau_intermediaire,unknown_total_intermediaire
0,DIEG2026-463,DA,MI-TEST-DA,1,False,0,False,1,False,1,False,0,False,0,False,1,False,1,False,1,False,0,False,1,False,0,False,1,False,0,False,0,False,8,53.333333,Intermédiaire,0
1,DIEG2026-636,DA,MI-TEST-DA,1,False,1,False,0,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,0,False,1,False,13,86.666667,Avancé,0
2,DIEG2026-633,DA,MI-TEST-DA,1,False,1,False,1,False,1,False,0,False,1,False,0,False,1,False,0,False,1,False,1,False,1,False,0,False,0,False,1,False,10,66.666667,Intermédiaire,0
3,DIEG2026-442,DA,MI-TEST-DA,1,False,0,False,1,False,1,False,0,False,1,False,1,False,1,False,0,False,0,False,1,False,1,False,0,False,0,False,0,False,8,53.333333,Intermédiaire,0
4,DIEG2026-055,DA,MI-TEST-DA,1,False,0,False,0,False,1,False,0,False,1,False,1,False,1,False,0,False,0,False,1,False,1,False,0,False,0,False,1,False,8,53.333333,Intermédiaire,0


# Cellule 12 — Calcul des scores des tests finaux /15

In [14]:
# ============================================================
# 2.11. Calcul des scores des tests finaux /15
# ============================================================

df_scores_finaux = df_tests_finaux[
    ["IDENTIFICATION", "parcours_test_final", "feuille_source_finale"]
].copy()

# Mapping du barème final
bareme_final = (
    df_correction_final
    .set_index(["parcours", "question_standardisee"])["bonne_reponse"]
    .to_dict()
)

# Calcul question par question
for q in colonnes_q_final:
    col_score = f"score_{q}"
    col_unknown = f"unknown_{q}"
    
    df_scores_finaux[col_score] = df_tests_finaux.apply(
        lambda row: corriger_reponse(
            row[q],
            bareme_final.get((row["parcours_test_final"], q))
        ),
        axis=1
    )
    
    df_scores_finaux[col_unknown] = df_tests_finaux[q].apply(
        est_reponse_inconnue
    )

# Score total /15
colonnes_scores_final = [f"score_{q}" for q in colonnes_q_final]
colonnes_unknown_final = [f"unknown_{q}" for q in colonnes_q_final]

df_scores_finaux["score_final_total"] = df_scores_finaux[colonnes_scores_final].sum(axis=1)

df_scores_finaux["score_final_pourcentage"] = (
    df_scores_finaux["score_final_total"] / 15 * 100
)

df_scores_finaux["niveau_final"] = df_scores_finaux[
    "score_final_pourcentage"
].apply(attribuer_niveau_depuis_pourcentage)

df_scores_finaux["unknown_total_final"] = df_scores_finaux[colonnes_unknown_final].sum(axis=1)

# Export
PATH_SCORES_FINAUX = OUTPUT_DIR / "03_scores_tests_finaux.csv"

df_scores_finaux.to_csv(
    PATH_SCORES_FINAUX,
    index=False,
    encoding="utf-8-sig"
)

print("Scores des tests finaux calculés.")
print("Fichier généré :", PATH_SCORES_FINAUX)
print("Dimensions :", df_scores_finaux.shape)

print("\nRésumé score final total /15 :")
display(df_scores_finaux["score_final_total"].describe())

print("\nRépartition des niveaux finaux :")
display(df_scores_finaux["niveau_final"].value_counts())

print("\nRépartition par parcours final :")
display(
    df_scores_finaux
    .groupby("parcours_test_final")["score_final_total"]
    .describe()
)

display(df_scores_finaux.head())

Scores des tests finaux calculés.
Fichier généré : outputs/02_scoring_pedagogique/03_scores_tests_finaux.csv
Dimensions : (206, 37)

Résumé score final total /15 :


count    206.000000
mean      11.004854
std        3.653929
min        0.000000
25%        8.000000
50%       12.000000
75%       14.000000
max       15.000000
Name: score_final_total, dtype: float64


Répartition des niveaux finaux :


niveau_final
Avancé           126
Débutant          41
Intermédiaire     39
Name: count, dtype: int64


Répartition par parcours final :


,count,mean,std,min,25%,50%,75%,max
parcours_test_final,,,,,,,,
BI,60.0,10.683333,3.486009,0.0,8.00,11.0,14.0,15.0
DA,34.0,11.676471,3.345921,2.0,10.25,13.0,14.0,15.0
DS,48.0,11.458333,3.524735,3.0,9.50,13.0,14.0,15.0
IA,64.0,10.609375,4.034050,1.0,7.00,12.0,14.0,15.0


,IDENTIFICATION,parcours_test_final,feuille_source_finale,score_final_Q01,unknown_final_Q01,score_final_Q02,unknown_final_Q02,score_final_Q03,unknown_final_Q03,score_final_Q04,unknown_final_Q04,score_final_Q05,unknown_final_Q05,score_final_Q06,unknown_final_Q06,score_final_Q07,unknown_final_Q07,score_final_Q08,unknown_final_Q08,score_final_Q09,unknown_final_Q09,score_final_Q10,unknown_final_Q10,score_final_Q11,unknown_final_Q11,score_final_Q12,unknown_final_Q12,score_final_Q13,unknown_final_Q13,score_final_Q14,unknown_final_Q14,score_final_Q15,unknown_final_Q15,score_final_total,score_final_pourcentage,niveau_final,unknown_total_final
0,DIEG2026-055,DA,FINAL-TEST-DA,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,0,False,0,False,0,False,0,False,0,False,0,False,0,False,8,53.333333,Intermédiaire,0
1,DIEG2026-117,DA,FINAL-TEST-DA,0,False,1,False,0,False,0,False,0,False,1,False,0,False,0,False,0,False,1,False,1,False,0,False,0,False,0,False,0,False,4,26.666667,Débutant,0
2,DIEG2026-195,DA,FINAL-TEST-DA,0,False,0,False,0,False,1,False,0,False,0,False,0,False,0,False,0,False,1,False,1,False,1,False,1,False,0,False,1,False,6,40.000000,Débutant,0
3,DIEG2026-278,DA,FINAL-TEST-DA,0,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,14,93.333333,Avancé,0
4,DIEG2026-288,DA,FINAL-TEST-DA,1,False,1,False,1,False,1,False,1,False,1,False,1,False,1,False,0,False,1,False,1,False,1,False,1,False,1,False,1,False,14,93.333333,Avancé,0


# Cellule 13 — Fusion des scores

Progression moyenne initial → final : +51,48 points
Réussite finale : 103 réussites / 25 non-réussites

In [15]:
# ============================================================
# 2.12. Fusion des scores initial, intermédiaire et final
# ============================================================

# Résumé score initial
colonnes_initial_resume = [
    "IDENTIFICATION",
    "score_initial_DA", "score_initial_DA_pct", "niveau_initial_DA",
    "score_initial_BI", "score_initial_BI_pct", "niveau_initial_BI",
    "score_initial_DS", "score_initial_DS_pct", "niveau_initial_DS",
    "score_initial_IA", "score_initial_IA_pct", "niveau_initial_IA",
    "score_initial_total",
    "score_initial_pourcentage",
    "niveau_initial_global",
    "unknown_total_initial"
]

df_scores_initial_resume = df_scores_initial[colonnes_initial_resume].copy()


# Résumé score intermédiaire
colonnes_mi_resume = [
    "IDENTIFICATION",
    "parcours_test_intermediaire",
    "score_intermediaire_total",
    "score_intermediaire_pourcentage",
    "niveau_intermediaire",
    "unknown_total_intermediaire"
]

df_scores_intermediaires_resume = df_scores_intermediaires[colonnes_mi_resume].copy()


# Résumé score final
colonnes_final_resume = [
    "IDENTIFICATION",
    "parcours_test_final",
    "score_final_total",
    "score_final_pourcentage",
    "niveau_final",
    "unknown_total_final"
]

df_scores_finaux_resume = df_scores_finaux[colonnes_final_resume].copy()


# ============================================================
# 1. Dataset orientation scoré : inscription + test initial
# ============================================================

df_dataset_orientation_score = df_orientation.merge(
    df_scores_initial_resume,
    on="IDENTIFICATION",
    how="inner",
    validate="one_to_one"
)


# ============================================================
# 2. Dataset prédiction finale scoré : initial + final
# ============================================================

df_dataset_prediction_score = (
    df_prediction
    .merge(df_scores_initial_resume, on="IDENTIFICATION", how="inner", validate="one_to_one")
    .merge(df_scores_finaux_resume, on="IDENTIFICATION", how="inner", validate="one_to_one")
)

df_dataset_prediction_score["progression_absolue_pct"] = (
    df_dataset_prediction_score["score_final_pourcentage"]
    - df_dataset_prediction_score["score_initial_pourcentage"]
)

df_dataset_prediction_score["reussite_finale"] = (
    df_dataset_prediction_score["score_final_pourcentage"] >= 50
).astype(int)


# ============================================================
# 3. Dataset longitudinal scoré : initial + intermédiaire + final
# ============================================================

df_dataset_longitudinal_score = (
    df_longitudinal
    .merge(df_scores_initial_resume, on="IDENTIFICATION", how="inner", validate="one_to_one")
    .merge(df_scores_intermediaires_resume, on="IDENTIFICATION", how="inner", validate="one_to_one")
    .merge(df_scores_finaux_resume, on="IDENTIFICATION", how="inner", validate="one_to_one")
)

df_dataset_longitudinal_score["progression_initial_finale_pct"] = (
    df_dataset_longitudinal_score["score_final_pourcentage"]
    - df_dataset_longitudinal_score["score_initial_pourcentage"]
)

df_dataset_longitudinal_score["progression_initial_intermediaire_pct"] = (
    df_dataset_longitudinal_score["score_intermediaire_pourcentage"]
    - df_dataset_longitudinal_score["score_initial_pourcentage"]
)

df_dataset_longitudinal_score["progression_intermediaire_finale_pct"] = (
    df_dataset_longitudinal_score["score_final_pourcentage"]
    - df_dataset_longitudinal_score["score_intermediaire_pourcentage"]
)

df_dataset_longitudinal_score["reussite_finale"] = (
    df_dataset_longitudinal_score["score_final_pourcentage"] >= 50
).astype(int)


# ============================================================
# 4. Export
# ============================================================

PATH_ORIENTATION_SCORE = OUTPUT_DIR / "04_dataset_orientation_score.csv"
PATH_PREDICTION_SCORE = OUTPUT_DIR / "05_dataset_prediction_score.csv"
PATH_LONGITUDINAL_SCORE = OUTPUT_DIR / "06_dataset_longitudinal_score.csv"

df_dataset_orientation_score.to_csv(PATH_ORIENTATION_SCORE, index=False, encoding="utf-8-sig")
df_dataset_prediction_score.to_csv(PATH_PREDICTION_SCORE, index=False, encoding="utf-8-sig")
df_dataset_longitudinal_score.to_csv(PATH_LONGITUDINAL_SCORE, index=False, encoding="utf-8-sig")

print("Fusion des scores terminée.")
print("Dataset orientation scoré :", df_dataset_orientation_score.shape)
print("Dataset prédiction scoré :", df_dataset_prediction_score.shape)
print("Dataset longitudinal scoré :", df_dataset_longitudinal_score.shape)

print("\nRésumé progression initiale → finale :")
display(df_dataset_prediction_score["progression_absolue_pct"].describe())

print("\nRéussite finale :")
display(df_dataset_prediction_score["reussite_finale"].value_counts())

display(df_dataset_prediction_score.head())

Fusion des scores terminée.
Dataset orientation scoré : (305, 58)
Dataset prédiction scoré : (128, 87)
Dataset longitudinal scoré : (125, 114)

Résumé progression initiale → finale :


count    128.000000
mean      51.484375
std       28.947137
min      -46.666667
25%       34.583333
50%       55.000000
75%       73.333333
max      100.000000
Name: progression_absolue_pct, dtype: float64


Réussite finale :


reussite_finale
1    103
0     25
Name: count, dtype: int64

,ins_CreatedAt,IDENTIFICATION,ins_Année de naissance,ins_Genre,ins_Région d'origine,ins_Filière,ins_Niveau d'étude,ins_Statut actuel,ins_Niveau informatique,ins_Niveau Excel,ins_Niveau Power BI,ins_Niveau Python,ins_Niveau IA,ins_Motivation,ins_Objectif pro,ins_Disponibilité,ins_Source info,ins_annee_naissance_originale,ins_annee_naissance,ins_age,init_CreatedAt,init_Date du test,init_Q1 - Resume ventes Excel,init_Q2 - Import CSV Excel,init_Q3 - Modele donnees Excel,init_Q4 - Mauvaise pratique visu Excel,init_Q5 - 2e grande valeur Excel,init_Q6 - Mesure vs Colonne DAX,init_Q7 - CA annee precedente DAX,init_Q8 - Vue Modele Power BI,init_Q9 - Acces directeurs regionaux,init_Q10 - 12 commerciaux 3 indicateurs,init_Q11 - Bibliotheque CSV Python,init_Q12 - Overfitting Underfitting,init_Q13 - Segmentation 50000 clients,init_Q14 - Deployer modele Python API,init_Q15 - Valeurs manquantes 30pc,init_Q16 - Role system prompt LLM,init_Q17 - Assistant IA PDF financiers,init_Q18 - Role embedding dans RAG,init_Q19 - Agent IA selection outil,init_Q20 - Sortie fiable LLM tableau,final_RowId,final_CreatedAt,final_InscriptionId,final_Filiere,final_Niveau d'Etude,final_Q01,final_Q02,final_Q03,final_Q04,final_Q05,final_Q06,final_Q07,final_Q08,final_Q09,final_Q10,final_Q11,final_Q12,final_Q13,final_Q14,final_Q15,final_parcours_test_final,final_feuille_source_finale,score_initial_DA,score_initial_DA_pct,niveau_initial_DA,score_initial_BI,score_initial_BI_pct,niveau_initial_BI,score_initial_DS,score_initial_DS_pct,niveau_initial_DS,score_initial_IA,score_initial_IA_pct,niveau_initial_IA,score_initial_total,score_initial_pourcentage,niveau_initial_global,unknown_total_initial,parcours_test_final,score_final_total,score_final_pourcentage,niveau_final,unknown_total_final,progression_absolue_pct,reussite_finale
0,2026-04-01 03:01:14+00:00,DIEG2026-003,2001,Feminin,Diana,Droit / Sciences politiques,Master 2 (M2),Diplome(e) en recherche d emploi,Aucun,Aucun,Aucun,Aucun,Aucun,"[""Ameliorer mon employabilite"", "" Acquerir des...",Entrepreneur,"[""En semaine - Matin""]",Facebook,2001,2001.0,25.0,2026-04-03 13:28:48+00:00,2026-04-03 13:28:47+00:00,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,15,2026-06-04 20:03:57+00:00,3,Droit / Sciences politiques,Master 2 (M2),"B) Extraire le fichier, nettoyer/transformer l...",A) Les mesures et cles vers les dimensions,B) Workspace / Espace de travail > Skills,A) Administration > Reglages > Integrations,C) Administration > Fonctions > Nouvelle fonction,D) Valider et structurer les donnees recues et...,D) Generer des graphiques interactifs ou expor...,C) Stocker et exposer par API les contenus/rap...,C) Les prompts temporaires non publies,C) Afficher les rapports et declencher/recuper...,C) Des nodes connectes qui executent des actio...,"B) Lancer l'ETL, appeler les API, publier dans...",A) Piloter ou modifier des workflows et integr...,"D) Verifier l'endpoint/API appele, le slug/id ...",D) Administration > Reglages > Connexion,IA,FINAL-TEST-IA,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,0,0.0,Débutant,20,IA,14,93.333333,Avancé,0,93.333333,1
1,2026-04-01 06:30:52+00:00,DIEG2026-007,2002,Masculin,Sofia,Autre,Master 2 (M2),Etudiant(e) en fin d etudes,Intermediaire,Debutant,Aucun,Intermediaire,Debutant,"[""Acquerir des competences techniques"", "" Real...",Developpeur Python / IA,"[""En semaine - Soir"", "" Week-end""]",WhatsApp,2002,2002.0,24.0,2026-04-01 06:41:29+00:00,2026-04-01 06:41:26+00:00,B. Tableau croise dynamique (TCD),C. Formules SI imbriquees,B. TCD simple,E. Je ne sais pas,B. GRANDE.VALEUR(plage;2),E. Je ne sais pas,E. Je ne sais pas,E. Je ne sais pas,B. Row-Level Security (RLS),C. 12 co

Les **3 datasets scorés** ont chacun un rôle différent :

```text
1. Dataset orientation scoré : 305 lignes | 58 colonnes
```

Utilisé pour **recommander le parcours adapté** à partir des données d’inscription + test initial.

Contient :

```text
profil apprenant + réponses initiales + scores initiaux + niveaux initiaux
```

---

```text
2. Dataset prédiction scoré : 128 lignes | 87 colonnes
```

Utilisé pour **entraîner les modèles ML** qui prédisent la performance finale.

Contient :

```text
profil + test initial + score final + niveau final + progression
```

---

```text
3. Dataset longitudinal scoré : 125 lignes | 114 colonnes
```

Utilisé pour analyser le parcours complet de l’apprenant.

Contient :

```text
inscription + test initial + test intermédiaire + test final + progression complète
```

Résumé simple :

```text
Orientation : pour recommander.
Prédiction : pour entraîner le modèle.
Longitudinal : pour analyser toute l’évolution.
```


# Cellule 14 — Validation finale des niveaux et export Étape 2

In [16]:
# ============================================================
# 2.13. Validation finale des niveaux et export Étape 2
# ============================================================

# Alias officiel demandé dans le mémoire
df_dataset_prediction_score["progression_absolue"] = df_dataset_prediction_score["progression_absolue_pct"]
df_dataset_longitudinal_score["progression_absolue"] = df_dataset_longitudinal_score["progression_initial_finale_pct"]

# Clusters officiels
df_dataset_orientation_score["cluster_initial"] = df_dataset_orientation_score["niveau_initial_global"]
df_dataset_prediction_score["cluster_initial"] = df_dataset_prediction_score["niveau_initial_global"]
df_dataset_prediction_score["cluster_final"] = df_dataset_prediction_score["niveau_final"]

df_dataset_longitudinal_score["cluster_initial"] = df_dataset_longitudinal_score["niveau_initial_global"]
df_dataset_longitudinal_score["cluster_intermediaire"] = df_dataset_longitudinal_score["niveau_intermediaire"]
df_dataset_longitudinal_score["cluster_final"] = df_dataset_longitudinal_score["niveau_final"]

# Fonction simple d'interprétation de progression
def qualifier_progression(valeur):
    if pd.isna(valeur):
        return np.nan
    elif valeur > 5:
        return "Progression"
    elif valeur < -5:
        return "Régression"
    else:
        return "Stable"

df_dataset_prediction_score["statut_progression"] = df_dataset_prediction_score["progression_absolue"].apply(qualifier_progression)
df_dataset_longitudinal_score["statut_progression"] = df_dataset_longitudinal_score["progression_absolue"].apply(qualifier_progression)

# Exports finaux Étape 2
df_dataset_orientation_score.to_csv(
    OUTPUT_DIR / "04_dataset_orientation_score.csv",
    index=False,
    encoding="utf-8-sig"
)

df_dataset_prediction_score.to_csv(
    OUTPUT_DIR / "05_dataset_prediction_score.csv",
    index=False,
    encoding="utf-8-sig"
)

df_dataset_longitudinal_score.to_csv(
    OUTPUT_DIR / "06_dataset_longitudinal_score.csv",
    index=False,
    encoding="utf-8-sig"
)

# Rapport synthèse
rapport_step2 = {
    "etape": "Étape 2 — Scoring pédagogique et calcul des niveaux",
    "test_initial": {
        "nb_apprenants": int(len(df_scores_initial)),
        "score_max": 20,
        "moyenne": float(df_scores_initial["score_initial_total"].mean()),
        "repartition_niveaux": df_scores_initial["niveau_initial_global"].value_counts().to_dict()
    },
    "test_intermediaire": {
        "nb_apprenants": int(len(df_scores_intermediaires)),
        "score_max": 15,
        "moyenne": float(df_scores_intermediaires["score_intermediaire_total"].mean()),
        "repartition_niveaux": df_scores_intermediaires["niveau_intermediaire"].value_counts().to_dict()
    },
    "test_final": {
        "nb_apprenants": int(len(df_scores_finaux)),
        "score_max": 15,
        "moyenne": float(df_scores_finaux["score_final_total"].mean()),
        "repartition_niveaux": df_scores_finaux["niveau_final"].value_counts().to_dict()
    },
    "datasets_scores": {
        "orientation": df_dataset_orientation_score.shape,
        "prediction": df_dataset_prediction_score.shape,
        "longitudinal": df_dataset_longitudinal_score.shape
    },
    "niveaux_officiels": ["Débutant", "Intermédiaire", "Avancé"],
    "statut": "Étape 2 validée"
}

with open(OUTPUT_DIR / "rapport_step2_scoring_pedagogique.json", "w", encoding="utf-8") as f:
    json.dump(rapport_step2, f, ensure_ascii=False, indent=4)

print("Étape 2 terminée et validée.")
print("Fichiers générés dans :", OUTPUT_DIR)

print("\nNiveaux initiaux :")
display(df_scores_initial["niveau_initial_global"].value_counts())

print("\nNiveaux intermédiaires :")
display(df_scores_intermediaires["niveau_intermediaire"].value_counts())

print("\nNiveaux finaux :")
display(df_scores_finaux["niveau_final"].value_counts())

print("\nProgression initiale → finale :")
display(df_dataset_prediction_score["statut_progression"].value_counts())

rapport_step2

Étape 2 terminée et validée.
Fichiers générés dans : outputs/02_scoring_pedagogique

Niveaux initiaux :


niveau_initial_global
Débutant         268
Intermédiaire     22
Avancé            15
Name: count, dtype: int64


Niveaux intermédiaires :


niveau_intermediaire
Avancé           108
Intermédiaire     54
Débutant          34
Name: count, dtype: int64


Niveaux finaux :


niveau_final
Avancé           126
Débutant          41
Intermédiaire     39
Name: count, dtype: int64


Progression initiale → finale :


statut_progression
Progression    118
Stable           6
Régression       4
Name: count, dtype: int64

{'etape': 'Étape 2 — Scoring pédagogique et calcul des niveaux',
 'test_initial': {'nb_apprenants': 305,
  'score_max': 20,
  'moyenne': 4.577049180327869,
  'repartition_niveaux': {'Débutant': 268, 'Intermédiaire': 22, 'Avancé': 15}},
 'test_intermediaire': {'nb_apprenants': 196,
  'score_max': 15,
  'moyenne': 10.811224489795919,
  'repartition_niveaux': {'Avancé': 108, 'Intermédiaire': 54, 'Débutant': 34}},
 'test_final': {'nb_apprenants': 206,
  'score_max': 15,
  'moyenne': 11.004854368932039,
  'repartition_niveaux': {'Avancé': 126, 'Débutant': 41, 'Intermédiaire': 39}},
 'datasets_scores': {'orientation': (305, 59),
  'prediction': (128, 91),
  'longitudinal': (125, 119)},
 'niveaux_officiels': ['Débutant', 'Intermédiaire', 'Avancé'],
 'statut': 'Étape 2 validée'}

Donc **Étape 2 — Scoring pédagogique et calcul des niveaux : TERMINÉE**.

Résultats principaux :

```text id="xkn5qm"
Test initial :
305 apprenants
Moyenne : 4,58 /20
Débutant : 268
Intermédiaire : 22
Avancé : 15
```

```text id="cn4scq"
Test intermédiaire :
196 apprenants
Moyenne : 10,81 /15
Avancé : 108
Intermédiaire : 54
Débutant : 34
```

```text id="izqtxn"
Test final :
206 apprenants
Moyenne : 11,00 /15
Avancé : 126
Intermédiaire : 39
Débutant : 41
```

Progression initiale → finale :

```text id="kp7xlw"
Progression : 118
Stable : 6
Régression : 4
```

Conclusion :

```text id="qj8pl4"
Étape 2 validée.
Les scores, niveaux et progressions sont maintenant calculés.
```

Prochaine étape logique :

```text id="kebvbt"
Étape 3 — Analyse exploratoire des scores et profils des apprenants
```

On analysera les distributions, les niveaux par parcours, les progressions et les premiers profils d’apprenants.
